# LegalGPT — Google Colab (No Firebase, with file attachments)\n\nThis notebook runs the included FastAPI backend, browser frontend, Ollama model, FAISS legal RAG search, and SQLite chat history inside the Colab runtime. The updated UI supports image, PDF, DOCX and legacy DOC attachments.\n\n**Limitations:** Colab is still cloud computing. The website works only while this runtime is running. Chat history is stored in the temporary Colab disk and can disappear when the runtime is deleted. There is no Google login or account-level privacy.\n\nUpload the ZIP file, then run the cells from top to bottom.\n

## 1. Upload and extract the ZIP

In [ ]:
from google.colab import files
import zipfile, os, shutil

uploaded = files.upload()  # Choose LegalGPT_Colab_No_Firebase.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')

PROJECT = '/content/LegalGPT_Colab_No_Firebase'
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Files:', os.listdir(PROJECT))


## 2. Install Python and system packages\n\nThe extra packages enable PDF/DOCX extraction and image OCR. `antiword` enables older `.doc` files.\n

In [ ]:
%pip install -q -r requirements.txt -r requirements-web.txt\n

In [ ]:
!apt-get -qq update && apt-get -qq install -y antiword tesseract-ocr

## 3. Install and start Ollama

Ollama runs inside this Colab runtime. It is not connected to an Ollama installation on your computer.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, os

ollama_log = open('/content/ollama.log', 'w')
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
time.sleep(8)
print('Ollama process running:', ollama_process.poll() is None)
print(open('/content/ollama.log').read()[-1500:])


In [ ]:
!ollama pull llama3.2:3b


In [ ]:
import requests
r = requests.get('http://127.0.0.1:11434/api/tags', timeout=20)
r.raise_for_status()
print('Ollama models:', [m['name'] for m in r.json().get('models', [])])


## 4. Check or rebuild the legal RAG index

In [ ]:
from pathlib import Path
required = ['dataset.csv','dataset_cleaned.csv','index/legal.faiss','index/metadata.json','index/index_config.json']
for f in required:
    print(f, 'OK' if (Path(PROJECT)/f).exists() else 'MISSING')


If the three index files exist and you have not changed `dataset.csv`, you can skip the next cell. If you replaced or updated the dataset, run it to rebuild the index. The dataset must contain `Jurisdictional Reference`, `Type`, and `Description` columns.


In [ ]:
!python Cleaned_data.py
!python build_index.py


## 5. Start the backend

In [ ]:
import subprocess, time, requests, os
server_log = open('/content/legalgpt-server.log', 'w')
server_process = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=PROJECT, stdout=server_log, stderr=subprocess.STDOUT
)
time.sleep(15)
try:
    r = requests.get('http://127.0.0.1:8000/healthz', timeout=10)
    print('Backend:', r.status_code, r.json())
except Exception:
    print(open('/content/legalgpt-server.log').read()[-5000:])
    raise


## 6. Open the website from Colab

This uses Colab's built-in port proxy to access the server running in this notebook. It does not create a permanent public website or use Cloudflare/ngrok. Open the link printed below in a new browser tab.


In [ ]:
from google.colab import output
web_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
print('Open your LegalGPT website:')
print(web_url)


## 7. Use the application

- Click **＋** beside the message box to attach PNG, JPG, JPEG, WEBP, PDF, DOCX or DOC files.
- Attach up to 5 files per message (15 MB each), add a question, and press Enter or Send.
- PDFs and Word files are text-extracted. Images use OCR by default.
- For full visual image understanding (not only text OCR), optionally install an Ollama vision model such as `llava:7b`, then set `OLLAMA_VISION_MODEL=llava:7b` before starting the server. Vision models can be large and may exceed available Colab memory.
- Click **New chat** to create another conversation. Previous chats appear in the left panel.
- Chat history and attachment names are saved in `legalgpt.sqlite3` inside this Colab runtime. Original uploaded files are processed in memory; image bytes are not retained in chat history.

## 8. Stop the application

When finished, stop the runtime from **Runtime → Disconnect and delete runtime**. This stops the model and backend. The temporary SQLite database may be deleted with the runtime.

## Troubleshooting

- **Ollama connection refused:** rerun the Ollama start cell and check `/content/ollama.log`.
- **Model missing:** run `!ollama pull llama3.2:3b` and restart the backend cell.
- **Could not read a legacy .doc:** make sure the antiword installation cell completed successfully.
- **Image answers only mention OCR text:** install a vision model and set `OLLAMA_VISION_MODEL` before starting Uvicorn.
